# Healthcare Data Lake — AWS S3 (Serverless)

Run cells 1 → 2 → 3 → 4 in order.

In [ ]:
%pip install boto3 --quiet

In [ ]:
import boto3, io, pandas as pd
from pyspark.sql.types import StructType, StructField, StringType

s3 = boto3.client(
    "s3",
    aws_access_key_id     = "YOUR_AWS_ACCESS_KEY_ID",
    aws_secret_access_key = "YOUR_AWS_SECRET_ACCESS_KEY",
    region_name           = "us-east-1"
)

# Quick connectivity test — should print 16 folder names
resp = s3.list_objects_v2(
    Bucket="healthcare-exports-005905648819",
    Prefix="datalake/",
    Delimiter="/"
)
for p in resp.get("CommonPrefixes", []):
    print(p["Prefix"])


In [ ]:
# Load all 16 tables as Spark temp views (reads live from S3, no copy)
def read_table(stem):
    obj = s3.get_object(
        Bucket="healthcare-exports-005905648819",
        Key=f"datalake/{stem}/{stem}.csv"
    )
    # dtype=str  => uniform string schema, no mixed-type inference
    # fillna("") => no NaN so Arrow bridge never chokes
    pdf = pd.read_csv(io.BytesIO(obj["Body"].read()), dtype=str).fillna("")
    # Explicit schema avoids CANNOT_INFER_EMPTY_SCHEMA on tables with 0 rows
    schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
    return spark.createDataFrame(pdf, schema=schema)

TABLES = [
    "allergies", "careplans", "conditions", "devices", "encounters",
    "imaging_studies", "immunizations", "medications", "observations",
    "organizations", "patients", "payer_transitions", "payers",
    "procedures", "providers", "supplies"
]

for name in TABLES:
    df = read_table(name)
    df.createOrReplaceTempView(name)
    print(f"  {name:25s}  {df.count():>10,} rows")


In [ ]:
display(spark.sql("SELECT id, first, last, gender, race, birthdate, city, state FROM patients LIMIT 20"))

In [ ]:
display(spark.sql("""
  SELECT description, COUNT(*) AS patient_count
  FROM conditions
  GROUP BY description
  ORDER BY patient_count DESC
  LIMIT 20
"""))

In [ ]:
display(spark.sql("""
  SELECT p.first, p.last, p.gender, p.race,
         COUNT(DISTINCT e.id)   AS encounters,
         COUNT(DISTINCT m.code) AS medications,
         COUNT(DISTINCT c.code) AS conditions
  FROM   patients p
  LEFT JOIN encounters  e ON e.patient = p.id
  LEFT JOIN medications m ON m.patient = p.id
  LEFT JOIN conditions  c ON c.patient = p.id
  GROUP BY p.first, p.last, p.gender, p.race
  ORDER BY encounters DESC
  LIMIT 20
"""))